In [ ]:
import numpy as np
from scipy.spatial import Delaunay
import pyomo.environ as pyo
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
%matplotlib widget

# ======== 真实函数（也是 obj_expr ）========
def f_true_numpy(xy):
    x, y = float(xy[0]), float(xy[1])
    return 2.0*(x - 3.0)**2 + 3.0*(y - 5.0)**2

def build_model():
    m = pyo.ConcreteModel()
    m.x = pyo.Var(bounds=(0.0, 10.0))
    m.y = pyo.Var(bounds=(0.0, 10.0))
    return m

def f_true_expr(m):
    return 2*(m.x - 3)**2 + 3*(m.y - 5)**2

def drop_simplex_stuff(m):
    for nm in ['lam','lam_sum','x_link_x','x_link_y','As','As_con','obj']:
        if hasattr(m, nm):
            m.del_component(nm)

# ======== 每个三角形里求 ms、LB、UB（复用同一个 model 实例）========
def minimize_on_each_triangle(nodes, values, solver='ipopt', tee=False):
    pts  = np.asarray(nodes, dtype=float)
    vals = np.asarray(values, dtype=float)
    tri = Delaunay(pts)

    m = build_model()
    opt = pyo.SolverFactory(solver)

    results = []
    for k, simp in enumerate(tri.simplices):
        verts  = pts[simp]     # (3,2)
        fverts = vals[simp]    # (3,)

        # 重心变量 lam_j >=0, sum lam =1
        m.lam = pyo.Var(range(3), domain=pyo.NonNegativeReals)
        m.lam_sum = pyo.Constraint(expr=sum(m.lam[j] for j in range(3)) == 1.0)

        # x = sum lam_j * v_jx,  y = sum lam_j * v_jy
        m.x_link_x = pyo.Constraint(expr=m.x == sum(m.lam[j]*float(verts[j,0]) for j in range(3)))
        m.x_link_y = pyo.Constraint(expr=m.y == sum(m.lam[j]*float(verts[j,1]) for j in range(3)))

        # As = sum lam_j * f(v_j)
        m.As = pyo.Var()
        m.As_con = pyo.Constraint(expr=m.As == sum(m.lam[j]*float(fverts[j]) for j in range(3)))

        # 目标：min f(x,y) - As
        m.obj = pyo.Objective(expr=f_true_expr(m) - m.As, sense=pyo.minimize)

        res = opt.solve(m, tee=tee)

        x_ms = np.array([pyo.value(m.x), pyo.value(m.y)], dtype=float)
        ms_val = float(pyo.value(m.obj))     # min_x [f - As]
        f_at_x_ms = f_true_numpy(x_ms)

        # LB = min As + ms = min f(顶点) + ms
        LB = float(np.min(fverts) + ms_val)
        # UB（报告用） = min( f(顶点), f(x_ms) )
        UB = float(min(np.min(fverts), f_at_x_ms))

        results.append({
            "simplex_index": int(k),
            "vertices_index": list(map(int, simp.tolist())),
            "verts": verts,
            "fverts": fverts,
            "x_ms": x_ms,
            "ms": ms_val,
            "LB": LB,
            "UB": UB,
            "f(x_ms)": f_at_x_ms,
            "status": str(res.solver.status),
            "term": str(res.solver.termination_condition),
        })

        drop_simplex_stuff(m)

    return tri, results

def compute_lb_min_node_max_ms(nodes, values, per_tri):
    idx_min = int(np.argmin(values))
    ms_candidates = [r['ms'] for r in per_tri if idx_min in r['vertices_index']]
    if ms_candidates:
        return float(values[idx_min] + max(ms_candidates)), idx_min, float(max(ms_candidates))
    fallback = float(min(r['LB'] for r in per_tri))
    return fallback, idx_min, float('nan')

# ======== NEW：在 active 单形中，按 As+ms 与 UB_global 的“等值线”裁边并生成新点 ========
def compute_edge_level_cuts(per_tri, UB_global, active_mask, min_dist=1e-6, tol_eq=1e-12):
    """
    按“无向边 key=(i,j) with i<j”聚合，每条边至多产生 1 个截点：
    - 若来自两个三角形都给了候选，保留 ms 较小那一个（更保守），也可改成取中点。
    """
    # 收集：edge_key -> [(pt, ms), ...]
    edge_hits = {}

    def add_hit(i, j, pt, ms):
        key = (i, j) if i < j else (j, i)
        edge_hits.setdefault(key, []).append((pt, ms))

    for r in per_tri:
        sid = r["simplex_index"]
        if not active_mask.get(sid, False):
            continue
        idx = r["vertices_index"]     # 全局顶点索引[3]
        verts = r["verts"]            # (3,2)
        fvs  = r["fverts"]            # (3,)
        z    = fvs + r["ms"]          # 抬高到 As+ms

        for (a, b) in ((0,1),(1,2),(2,0)):
            ia, ib = int(idx[a]), int(idx[b])
            za, zb = float(z[a]), float(z[b])
            va, vb = verts[a], verts[b]

            # 端点恰好在 UB
            if abs(za - UB_global) <= tol_eq:
                add_hit(ia, ib, va.copy(), r["ms"])
            if abs(zb - UB_global) <= tol_eq:
                add_hit(ia, ib, vb.copy(), r["ms"])

            prod = (za - UB_global) * (zb - UB_global)
            if prod < 0.0:  # 真正跨越
                t = (UB_global - za) / (zb - za)
                p = (1.0 - t) * va + t * vb
                add_hit(ia, ib, p.astype(float), r["ms"])

    # 每条边只取一个点：默认取 ms 较小（更保守）
    cuts = []
    for key, cand_list in edge_hits.items():
        if not cand_list:
            continue
        pt, _ms = max(cand_list, key=lambda w: w[1])  # 也可改成：np.mean([w[0] for w in cand_list], axis=0)
        # 与已有 cuts 基于 min_dist 再去重
        if all(np.linalg.norm(pt - q) >= min_dist for q in cuts):
            cuts.append(pt)
    return cuts


# ======== 选 x_ms 候选点（按 ms 排序，默认 ms 最小优先；过近就尝试下一个）========
def pick_candidate_by_ms_rank(per_tri, nodes, min_dist=1e-6, descending=False):
    cands = sorted(per_tri, key=lambda r: r['ms'], reverse=descending)
    def ok(p):
        return all(np.linalg.norm(p - q) >= min_dist for q in nodes)
    for rank, r in enumerate(cands, start=1):
        p = np.array(r['x_ms'], float)
        if ok(p):
            return p, rank, float(r['ms'])
    return None, None, None

# ======== 打印“列=三角形”的表格（保留）========
def print_per_triangle_table(results, active_mask):
    results = sorted(results, key=lambda r: r["simplex_index"])
    simplex_ids = [r["simplex_index"] for r in results]
    active_set = {sid for sid in simplex_ids if active_mask[sid]}

    header = ["row\\simp"] + [f"T{sid}{'*' if sid in active_set else ''}" for sid in simplex_ids]
    rows = [
        ["UB"] + [f"{r['UB']:.6f}" for r in results],
        ["LB"] + [f"{r['LB']:.6f}" for r in results],
        ["ms"] + [f"{r['ms']:.3e}" for r in results],
    ]
    table = [header] + rows
    ncols = len(header)
    colw = [0]*ncols
    for c in range(ncols):
        colw[c] = max(len(str(row[c])) for row in table) + 2

    RED = "\033[31m"; RESET = "\033[0m"
    def colorize_if_active(col_idx: int, s: str) -> str:
        if col_idx == 0: return s
        sid = simplex_ids[col_idx-1]
        return f"{RED}{s}{RESET}" if sid in active_set else s

    header_line = "".join(str(header[c]).ljust(colw[c]) if c == 0
                          else colorize_if_active(c, str(header[c]).ljust(colw[c]))
                          for c in range(ncols))
    print("\n== Per-triangle summary ==")
    print(header_line)
    print("-" * sum(colw))
    for r in rows:
        line = []
        for c in range(ncols):
            cell = str(r[c])
            padded = cell.ljust(colw[c]) if c == 0 else cell.rjust(colw[c])
            line.append(colorize_if_active(c, padded))
        print("".join(line))
    print("(红色列 = active；第1行=UB，第2行=LB，第3行=ms)\n")

# === Active 判定（方案一）：只保留下侧或穿越 UB 的单形 ===
def is_active_triangle(record, UB, tol):
    """
    record: per_tri 中的字典 r
    UB: 当前 UB_global
    tol: active_tol
    规则：
      - strictly below:  z_max < UB - tol
      - straddle:        z_min < UB - tol  且  z_max > UB + tol
      - 否则 inactive（尤其 z_min≈UB 且其余在上侧的，剔除）
    """
    zverts = record["fverts"] + record["ms"]    # 顶点抬高到 As+ms
    zmin   = float(np.min(zverts))
    zmax   = float(np.max(zverts))
    strictly_below = (zmax < UB - tol)
    straddle       = (zmin < UB - tol) and (zmax > UB + tol)
    return strictly_below or straddle

# ======== 绘图（保留）========
def plot_iteration(iter_id, nodes, tri, active_mask,
                   grid_n=60, elev=35, azim=-135,
                   min_node=None, new_node=None,
                   per_tri=None):
    nodes = np.asarray(nodes, float)

    COLOR_SURFACE = 'C0'
    COLOR_NODE_EDGES = '#444444'
    COLOR_PIECE = '#ff7043'
    LW_MAIN = 2.2

    xs = np.linspace(0, 10, grid_n); ys = np.linspace(0, 10, grid_n)
    XX, YY = np.meshgrid(xs, ys)
    ZZ = 2.0*(XX-3.0)**2 + 3.0*(YY-5.0)**2

    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111, projection='3d')
    ax.plot_surface(XX, YY, ZZ, alpha=0.35, linewidth=0, antialiased=True, color=COLOR_SURFACE)

    if per_tri is not None and len(per_tri) > 0 and tri is not None:
        faces = []
        for r in per_tri:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False): continue
            verts_xy = r["verts"]; fverts = r["fverts"]; ms_val = r["ms"]
            tri_xyz  = [(verts_xy[i,0], verts_xy[i,1], float(fverts[i] + ms_val)) for i in range(3)]
            faces.append(tri_xyz)
        if faces:
            coll = Poly3DCollection(faces, facecolors=[COLOR_PIECE],
                                    edgecolors='none', alpha=0.55)
            ax.add_collection3d(coll)
        for r in per_tri:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False): continue
            verts_xy = r["verts"]; fverts = r["fverts"]; ms_val = r["ms"]
            for i in range(3):
                a_xy = verts_xy[i]; b_xy = verts_xy[(i+1) % 3]
                za = float(fverts[i] + ms_val); zb = float(fverts[(i+1) % 3] + ms_val)
                ax.plot([a_xy[0], b_xy[0]], [a_xy[1], b_xy[1]], [za, zb], lw=LW_MAIN, c=COLOR_PIECE)

    node_z = np.array([f_true_numpy(p) for p in nodes])
    ax.scatter(nodes[:,0], nodes[:,1], node_z, s=30, c='k', depthshade=False)

    if tri is not None:
        for k, simp in enumerate(tri.simplices):
            if not active_mask.get(k, False): continue
            verts_xy = tri.points[simp]
            for i in range(3):
                a, b = verts_xy[i], verts_xy[(i+1) % 3]
                ax.plot([a[0], b[0]], [a[1], b[1]],
                        [f_true_numpy(a), f_true_numpy(b)], lw=LW_MAIN, c=COLOR_NODE_EDGES)
            centroid = np.mean(verts_xy, axis=0)
            ax.scatter([centroid[0]], [centroid[1]], [f_true_numpy(centroid)],
                       marker='^', c=COLOR_NODE_EDGES, s=70, depthshade=False)

    if min_node is not None:
        ax.scatter([min_node[0]], [min_node[1]], [f_true_numpy(min_node)],
                   color='g', s=80, marker='o', label='Current min node')
    if new_node is not None:
        ax.scatter([new_node[0]], [new_node[1]], [f_true_numpy(new_node)],
                   color='b', s=80, marker='o', label='Next node')

    ax.set_title(f"Iteration {iter_id}")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("f(x,y)")
    ax.legend(loc='upper left'); ax.view_init(elev=elev, azim=azim)
    plt.tight_layout(); plt.show()

def plot_convergence(hist, prec=4):
    x  = hist["node_count"]; LB = hist["LB_hist"]; UB = hist["UB_hist"]; ms = hist["ms_hist"]
    plt.figure(figsize=(6,4))
    plt.plot(x, LB, marker='o', label="LB (lower bound)")
    plt.plot(x, UB, marker='s', label="UB (upper bound)")
    plt.xlabel("Number of nodes"); plt.ylabel("Bound value")
    plt.title("Global LB and UB vs Number of Nodes"); plt.legend(); plt.grid(True); plt.tight_layout()

    plt.figure(figsize=(6,4))
    plt.plot(x, ms, marker='o', color='C2')
    plt.xlabel("Number of nodes"); plt.ylabel("Global ms (min over active simplices)")
    plt.title("ms vs Number of Nodes (Simplex method)")
    plt.grid(True); plt.tight_layout(); plt.show()

    print("\n==== Summary Table ====")
    fmt = f"{{:.{prec}f}}"; header = f"{'Nodes':>8s} | {'ms':>12s} | {'LB':>12s} | {'UB':>12s}"
    print(header); print("-" * len(header))
    for n, m, lb, ub in zip(x, ms, LB, UB):
        print(f"{n:8d} | {fmt.format(m):>12s} | {fmt.format(lb):>12s} | {fmt.format(ub):>12s}")
    print("-" * len(header)); print(f"{'Total':>8s} | {len(x):>12d}")

# ======== 主循环（两阶段：①按 UB 截面收缩并补点；②再按 ms 插值补点）========
def run_case(max_nodes=10, solver='ipopt', tee=False, plot_grid_n=60,
             record_history=True, make_3d_plots=True,
             only_from_active=True, min_dist=1e-4, active_tol=1e-8):
    """
    迭代顺序：
      A. 基于当前 nodes→Delaunay：求每个单形的 ms/LB/UB，判定 active；
      B. 在 active 单形上按 As+ms=UB_global 切边，生成“截面点”加入节点集；
      C. 用新节点集重建 Delaunay，再求一次每单形 ms/LB/UB；
      D. 在 candidates（默认为 active 集合）中按 ms 最小挑 x_ms 补 1 个点（若过近，尝试次小）。
    """
    # 初始角点
    nodes = [np.array([0.0,0.0]), np.array([0.0,10.0]),
             np.array([10.0,0.0]), np.array([10.0,10.0])]
    values = [f_true_numpy(p) for p in nodes]

    LB_hist, UB_hist, ms_hist, node_count = [], [], [], []

    it = 0
    while len(nodes) < max_nodes:
        # ---------- 第一次评估：基于当前 nodes ----------
        tri, per_tri = minimize_on_each_triangle(nodes, values, solver=solver, tee=tee)
        UB_global = float(np.min(values))
        LB_iter, idx_min_node, _ = compute_lb_min_node_max_ms(nodes, values, per_tri)
        ms_iter = min(r['ms'] for r in per_tri)
        active_mask = {
            r['simplex_index']: is_active_triangle(r, UB_global, active_tol)
            for r in per_tri
        }


        # 打印：本轮 UB + 表格
        print(f"\n[Iter {it}] UB_global = {UB_global:.6f}  (current best node value)")
        print_per_triangle_table(per_tri, active_mask)

        # ---------- A→B：按 UB 截面收缩并补点 ----------
        cut_points = compute_edge_level_cuts(per_tri, UB_global, active_mask,
                                             min_dist=min_dist, tol_eq=1e-12)
        n_before = len(nodes)
        # 合并并去重
        for p in cut_points:
            if all(np.linalg.norm(p - q) >= min_dist for q in nodes):
                nodes.append(p)
                values.append(f_true_numpy(p))
        n_after_cuts = len(nodes)
        num_new_from_cuts = n_after_cuts - n_before
        num_active = sum(1 for v in active_mask.values() if v)
        print(f"[Iter {it}] active simplices: {num_active}, "
              f"cut edges generated raw pts: {len(cut_points)}, "
              f"unique new nodes added by cuts: {num_new_from_cuts}")

        # ---------- B→C：用“收缩补点后”的 nodes 再评估一次 ----------
        tri2, per_tri2 = minimize_on_each_triangle(nodes, values, solver=solver, tee=tee)
        UB_global2 = float(np.min(values))
        LB_iter2, idx_min_node2, _ = compute_lb_min_node_max_ms(nodes, values, per_tri2)
        ms_iter2 = min(r['ms'] for r in per_tri2)
        active_mask2 = {
            r['simplex_index']: is_active_triangle(r, UB_global2, active_tol)
            for r in per_tri2
        }

        # 记录历史（使用第二次评估后的指标更贴近收缩后的状态）
        if record_history:
            LB_hist.append(LB_iter2)
            UB_hist.append(UB_global2)
            ms_hist.append(ms_iter2)
            node_count.append(len(nodes))

        # 可视化（基于第二次评估的单形）
        if make_3d_plots:
            plot_iteration(it, nodes, tri2, active_mask2,
                           grid_n=plot_grid_n,
                           min_node=nodes[idx_min_node2],
                           new_node=None, per_tri=per_tri2)

        # ---------- C→D：在（收缩后的）候选里按 ms 最小挑 x_ms 补 1 个点 ----------
        candidates = [r for r in per_tri2 if (not only_from_active) or active_mask2.get(r['simplex_index'], False)]
        if only_from_active and len(candidates) == 0:
            print(f"[warn] No active simplices after cuts (tol={active_tol:g}); fall back to all simplices.")
            candidates = per_tri2

        new_node, used_rank, used_ms = pick_candidate_by_ms_rank(
            candidates, nodes, min_dist=min_dist, descending=False
        )
        if new_node is None:
            print("[stop] All x_ms candidates are too close to existing nodes.")
            break

        # 最终把 x_ms 也加入一次
        nodes.append(new_node)
        values.append(f_true_numpy(new_node))
        print(f"[Iter {it}] added x_ms node with ms={used_ms:.3e} (rank={used_rank})")

        # 若还想在同一图里标蓝点，可再画一次（可选）
        if make_3d_plots:
            plot_iteration(it, nodes, tri2, active_mask2,
                           grid_n=plot_grid_n,
                           min_node=nodes[idx_min_node2],
                           new_node=new_node, per_tri=per_tri2)

        it += 1
        if len(nodes) >= max_nodes:
            break

    return {
        "nodes": np.array(nodes), "values": np.array(values),
        "LB_hist": LB_hist, "UB_hist": UB_hist, "ms_hist": ms_hist,
        "node_count": node_count
    }

# ======== 调用示例 ========
if __name__ == "__main__":
    solver      = "gurobi"   # 或 "gurobi"
    max_nodes   = 50
    plot_3d     = True
    record_hist = True

    hist_simp = run_case(
        max_nodes=max_nodes,
        solver=solver,
        tee=False,
        plot_grid_n=60,
        record_history=record_hist,
        make_3d_plots=plot_3d,
        only_from_active=True,
        min_dist=1e-4,
        active_tol=1e-8
    )

    plot_convergence(hist_simp, prec=4)

    nodes  = hist_simp["nodes"]
    values = hist_simp["values"]
    print("\n==== Done ====")
    print(f"Total nodes: {len(nodes)}")
    print(f"Best f among nodes: {np.min(values):.6f}")


ipopt


ApplicationError: No executable found for solver 'ipopt'